In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/02_silver_cleaning/00_common_functions

In [0]:
df = spark.read.table("hive_metastore.bronze.bronze_arqlmed")
display(df.limit(5))
df.printSchema()

In [0]:
df = parse_ts(df, "DATA_")

display(df)

In [0]:
df = df.drop("CIM_UID", "OPR_STM_FONTE", "OPR_ID_LOAD", "OPR_TM_LOAD", "OPR_TIPO_OPERACAO", "POLO", "NTIME", "DATA_")

In [0]:
df = cast_manual(df, "STATE", 'double')


In [0]:
df = df.withColumn("STATE", round("STATE", 2))

In [0]:
dbutils.data.summarize(df)

897M de registos

min: 2023-07-04T10:25:19Z

max: 2024-07-03T00:00:38Z

In [0]:
df = df.filter(
    (col("ID").substr(2, 1).isin("P", "S")) &  # Check position 2
    (~col("ID").substr(7, 1).isin("-", "9", "4"))  # Check position 7
)

In [0]:
# Generate df_arqlmed_U- using filter and like operations
filtered_df = df.filter((col("ID").rlike("U--$")) | (col("ID").rlike("0II--$")))


In [0]:
filtered_df = filtered_df.filter(filtered_df["STATE"] >= 0)

In [0]:
dbutils.data.summarize(filtered_df)

Temos agora 383M de valores

In [0]:
# data inicio dados = 2023-11-07 data fim = 2023-12-03

# Specify the equipment ID and metric you want to plot
equipment_id = "QSVLN-5505-0TU--"
metric_column = "STATE"
start_date = "2023-07-04"  # Start date for filtering
end_date = "2024-07-03"    # End date for filtering (for a week-long span)

# Filter the DataFrame for the specified equipment ID and date range
graph_df = filtered_df.filter((filtered_df["ID"] == equipment_id) &
                                 (filtered_df["DATE"] >= start_date) &
                                 (filtered_df["DATE"] <= end_date))

# Sort the DataFrame by the timestamp
sorted_df = graph_df.orderBy("DATE")

# Convert PySpark DataFrame to Pandas DataFrame
pd_df = sorted_df.select("DATE", metric_column).toPandas()


# Plot the data using Plotly
fig = px.line(pd_df, x="DATE", y=metric_column, title=f"Metric {metric_column} for Equipment ID {equipment_id}")

# Add a horizontal line representing the limit
fig.add_shape(
    type="line",
    x0=pd_df["DATE"].min(),
    x1=pd_df["DATE"].max(),
    line=dict(color="Red", width=2, dash="dash"),
    name="High Limit"
)

# Update layout to include the limit in the legend
fig.update_layout(
    shapes=[dict(
        type="line",
        x0=pd_df["DATE"].min(),
        x1=pd_df["DATE"].max(),
        line=dict(color="Red", width=2, dash="dash")
    )],
    annotations=[dict(
        x=pd_df["DATE"].mean(),
        xref="x",
        yref="y",
        showarrow=False,
        font=dict(color="Red")
    )]
)

# Show the plot
fig.show()

Pivoting the ID column

In [0]:
df_a = filtered_df.withColumn("ID_prefix", substring(col("ID"), 1, 11))

In [0]:
# Group by the first 12 characters and filter groups with more than one distinct ID
temp_df = df_a.groupBy("ID_prefix").agg(count_distinct("ID").alias("distinct_count")) \
    .filter(col("distinct_count") > 1)

# Join back with the original DataFrame to filter the relevant rows
df_a = df_a.join(temp_df, "ID_prefix")

# Show the result
df_a.display()

In [0]:
interval_s = 15 * 60  # 900

df_a = df_a.withColumn(
    "DATE_15M",
    F.to_timestamp(
        F.from_unixtime(
            (F.round(F.unix_timestamp(col("DATE")) / interval_s) * interval_s).cast("long")
        )
    )
)

display(df_a.select("DATE", "DATE_15M").limit(20))


In [0]:
df_a = (
    df_a
    .drop("DATE")
    .withColumnRenamed("DATE_15M", "DATE")
)

In [0]:
display(
    df_a.select(
        ((F.unix_timestamp(col("DATE")) % 900)).alias("offset_s")
    )
    .groupBy("offset_s")
    .count()
    .orderBy("offset_s")
)


In [0]:
p_df = df_a.withColumn("ID_prefix", substring(col("ID"), 1, 12)) \
           .withColumn("suffix", substring(col("ID"), -3, 3))

# List of columns to pivot
columns_to_pivot = ["STATE"]


pivoted_dfs = []
for column in columns_to_pivot:
    pivoted_df = p_df.groupBy("ID_prefix", "DATE").pivot("suffix").agg(F.first(column))
    
    # Check the column names in the pivoted DataFrame
    print(pivoted_df.columns)
    
    # Rename columns based on expected pivot values
    pivoted_df = pivoted_df.withColumnRenamed("U--", f"{column}_T").withColumnRenamed("I--", f"{column}_I")
    
    # Ensure that renaming reflects actual column names after pivot
    if 'STATE_T' in pivoted_df.columns and 'STATE_I' in pivoted_df.columns:
        pivoted_df = pivoted_df.withColumnRenamed("STATE_T", "VOLTAGE").withColumnRenamed("STATE_I", "CURRENT")
    
    pivoted_dfs.append(pivoted_df)

In [0]:
display(pivoted_df)

Dealing with outliers

In [0]:
null_rows = pivoted_df.filter(
    F.col("current").isNull() | F.col("voltage").isNull()
)

display(null_rows)

In [0]:
display(df.filter(col("ID") == "LSPOV-3318-0TU--"))

In [0]:
display(pivoted_df.filter(col("CURRENT") > 1000))

In [0]:
min_rows = 34000
max_null_rate = 0.01  # try 0.01 first, change to 0.03 if you will fill later

id_stats = (
    pivoted_df
    .groupBy("ID_prefix")
    .agg(
        F.count("*").alias("n_rows"),
        F.sum((F.col("current").isNull() | F.col("voltage").isNull()).cast("int")).alias("n_null_rows")
    )
    .withColumn("null_rate", F.col("n_null_rows") / F.col("n_rows"))
    .orderBy(F.desc("n_null_rows"))
)

display(id_stats)

In [0]:
good_ids = (
    id_stats
    .filter((F.col("n_rows") >= min_rows) & (F.col("null_rate") <= max_null_rate))
    .select("ID_prefix")
)

pivoted_df = pivoted_df.join(good_ids, on="ID_prefix", how="left_semi")

In [0]:
id_stats = (
    pivoted_df
    .groupBy("ID_prefix")
    .agg(
        F.count("*").alias("n_rows"),
        F.sum((F.col("current").isNull() | F.col("voltage").isNull()).cast("int")).alias("n_null_rows")
    )
    .withColumn("null_rate", F.col("n_null_rows") / F.col("n_rows"))
    .orderBy(F.desc("n_null_rows"))
)

display(id_stats)

In [0]:
# set these to your real column names
id_col = "ID_prefix"
ts_col = "DATE"

w_ffill = (
    Window.partitionBy(id_col)
          .orderBy(F.col(ts_col))
          .rowsBetween(Window.unboundedPreceding, 0)
)

pivoted_df = (
    pivoted_df
    .withColumn("current", F.last("current", ignorenulls=True).over(w_ffill))
    .withColumn("voltage", F.last("voltage", ignorenulls=True).over(w_ffill))
)

display(pivoted_df.select(id_col, ts_col, "current", "voltage").orderBy(id_col, ts_col).limit(50))

In [0]:
id_stats = (
    pivoted_df
    .groupBy("ID_prefix")
    .agg(
        F.count("*").alias("n_rows"),
        F.sum((F.col("current").isNull() | F.col("voltage").isNull()).cast("int")).alias("n_null_rows")
    )
    .withColumn("null_rate", F.col("n_null_rows") / F.col("n_rows"))
    .orderBy(F.desc("n_null_rows"))
)

display(id_stats)

In [0]:
display(pivoted_df.filter(col("CURRENT") > 1000))

In [0]:
display(pivoted_df.filter(col("ID_prefix") == "LPFANH5515-0").filter(col("current") > 300))

Creating Rolling Statistics

In [0]:
id_col = "ID_prefix"   # change to your id column, for example "ID"
ts_col = "DATE"        # your corrected 15-min timestamp column

windows = {
    "1h": 4,
    "1d": 96,
    "7d": 672
}

metrics_cols = ["current", "voltage"]

In [0]:
df_roll = pivoted_df.orderBy(id_col, ts_col)

for w_name, n_rows in windows.items():
    w = (
        Window.partitionBy(id_col)
              .orderBy(F.col(ts_col))
              .rowsBetween(-n_rows, -1)
    )

    for c in metrics_cols:
        df_roll = (
            df_roll
            .withColumn(f"{c}_mean_{w_name}", F.avg(F.col(c)).over(w))
            .withColumn(f"{c}_std_{w_name}", F.stddev(F.col(c)).over(w))
            .withColumn(f"{c}_max_{w_name}", F.max(F.col(c)).over(w))
        )


for w_name in windows.keys():
    for c in metrics_cols:
        for stat in ["mean", "std", "max"]:
            col_roll = f"{c}_{stat}_{w_name}"
            if col_roll in df_roll.columns:
                df_roll = df_roll.withColumn(col_roll, F.round(F.col(col_roll), 2))



display(df_roll.limit(20))


In [0]:
for w_name in windows.keys():
    for c in metrics_cols:
        for stat in ["mean", "std", "max"]:
            col_roll = f"{c}_{stat}_{w_name}"
            if col_roll in df_roll.columns:
                df_roll = df_roll.withColumn(col_roll, F.coalesce(F.col(col_roll), F.col(c)))

display(df_roll)


In [0]:
ts_col = "DATE"

daily_avg = (
    pivoted_df
    .groupBy(F.date_trunc("day", F.col(ts_col)).alias("day"))
    .agg(
        F.avg("voltage").alias("avg_voltage"),
        F.avg("current").alias("avg_current")
    )
    .orderBy("day")
)

display(daily_avg)


In [0]:
pdf = daily_avg.toPandas()
pdf["day"] = pd.to_datetime(pdf["day"])


In [0]:
plt.figure(figsize=(14, 5))
plt.plot(pdf["day"], pdf["avg_voltage"], label="Avg Voltage")
plt.plot(pdf["day"], pdf["avg_current"], label="Avg Current")

plt.title("Daily average voltage and current (year)")
plt.xlabel("Date")
plt.ylabel("Average value")
plt.legend()
display(plt.gcf())


In [0]:
display(df_roll.filter(col("CURRENT") > 400))

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "silver"
target_table = "silver_arqlmed"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

In [0]:
(
    df_roll.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))


In [0]:
df = spark.read.table("hive_metastore.silver.silver_arqlmed")
display(df.limit(5))
df.printSchema()

In [0]:
dbutils.data.summarize(df)